# Data Science Interview — Reference Solution Notebook
### Confidential (interviewer copy)

This notebook is the **model answer** for Parts 2 & 3 of the exercise. It shows what a strong
candidate's approach looks like and, in the commentary, *what signal each step is meant to reveal*.

Data: a slice of Amazon Sponsored Products performance data (`ad_performance.csv`, `campaigns.csv`).

**Metric definitions**
- CTR = clicks / impressions
- CVR = orders / clicks
- ACoS = spend / sales  (lower = more efficient)
- CPC  = spend / clicks


## 0. Setup

In [1]:
import numpy as np
import pandas as pd

pd.set_option("display.float_format", lambda x: f"{x:,.3f}")

# adjust paths if needed
PERF_PATH = "ad_performance.csv"
CAMP_PATH = "campaigns.csv"


## Part 2.1 — Load & clean

The point of this section is **judgment**, not just calling cleaning functions. Every issue gets
*investigated* before it gets *actioned*: some are fixed, some are excluded from a specific metric,
and one (wasted spend) is actually a business finding rather than dirt to throw away.

In [2]:
perf = pd.read_csv(PERF_PATH, parse_dates=["date"])
camp = pd.read_csv(CAMP_PATH, parse_dates=["launch_date"])

print("performance:", perf.shape)
print("campaigns:  ", camp.shape)
perf.head()

performance: (12651, 11)
campaigns:   (23, 6)


,date,campaign_id,keyword_id,keyword_text,match_type,impressions,clicks,spend,orders,sales,bid
0,2024-05-19,1005,5060,gift office,broad,95,0,0.000,0,0.000,0.710
1,2024-05-14,1022,5205,eco outdoor,phrase,97,4,3.580,1,17.930,0.890
2,2024-06-25,1000,5008,mini outdoor,exact,87,0,0.000,0,0.000,0.780
3,2024-06-06,1000,5005,deluxe outdoor,phrase,240,0,0.000,0,0.000,1.900
4,2024-06-02,1019,5180,starter kitchen,exact,167,2,2.320,0,0.000,1.160


In [3]:
# Always look before cleaning. describe() surfaces impossible values;
# isna() surfaces missingness.
display(perf.describe(include="all").T)
print("\nNulls per column:\n", perf.isna().sum())

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
date,12651,NaN,NaN,NaN,2024-05-30 11:57:05.847759,2024-05-01 00:00:00,2024-05-16 00:00:00,2024-05-31 00:00:00,2024-06-14 00:00:00,2024-06-29 00:00:00,NaN
campaign_id,"12,651.000",NaN,NaN,NaN,"1,010.381","1,000.000","1,005.000","1,010.000","1,016.000","1,022.000",6.625
keyword_id,"12,651.000",NaN,NaN,NaN,"5,104.469","5,000.000","5,052.000","5,104.000","5,157.000","5,209.000",60.671
keyword_text,12651,91,eco outdoor,424,NaN,NaN,NaN,NaN,NaN,NaN,NaN
match_type,12651,3,exact,4766,NaN,NaN,NaN,NaN,NaN,NaN,NaN
impressions,"12,651.000",NaN,NaN,NaN,104.866,0.000,69.000,103.000,139.000,283.000,50.615
clicks,"12,651.000",NaN,NaN,NaN,1.020,0.000,0.000,1.000,2.000,9.000,1.209
spend,"12,651.000",NaN,NaN,NaN,1.065,-1.300,0.000,0.610,1.500,15.630,1.513
orders,"12,651.000",NaN,NaN,NaN,0.108,0.000,0.000,0.000,0.000,4.000,0.349
sales,"12,525.000",NaN,NaN,NaN,3.033,0.000,0.000,0.000,0.000,107.870,10.119



Nulls per column:
 date              0
campaign_id       0
keyword_id        0
keyword_text      0
match_type        0
impressions       0
clicks            0
spend             0
orders            0
sales           126
bid              63
dtype: int64


**What to notice in the output above**
- `spend` has a **negative minimum** → refund artifacts.
- `sales` and `bid` have **nulls**.
- Some rows have `impressions == 0` yet positive `spend` → billing glitch.
- Row count is slightly inflated by **duplicate rows**.

In [4]:
# --- Issue 1: exact duplicate rows -> safe to drop ---
n_dupes = perf.duplicated().sum()
print("Full-row duplicates:", n_dupes)
perf = perf.drop_duplicates().reset_index(drop=True)

# --- Issue 2: negative spend (refunds) -> investigate, then exclude from efficiency ---
neg = perf[perf.spend < 0]
print("Negative-spend rows:", len(neg))

# --- Issue 3: spend but no traffic (0 impressions) -> a FINDING (wasted spend), not just dirt ---
glitch = perf[(perf.impressions == 0) & (perf.spend > 0)]
print("Zero-impression rows with spend:", len(glitch),
      "| wasted spend $%.2f" % glitch.spend.sum())

# --- Issue 4: nulls ---
print("Null sales:", perf.sales.isna().sum(), "| Null bid:", perf.bid.isna().sum())

Full-row duplicates: 62
Negative-spend rows: 5
Zero-impression rows with spend: 40 | wasted spend $106.92
Null sales: 126 | Null bid: 63


**Decisions (and the reasoning a strong candidate voices):**

| Issue | Decision | Why |
|---|---|---|
| Duplicate rows | Drop | Exact dupes double-count spend/sales; clearly wrong. |
| Negative spend | Exclude from efficiency metrics | Refund artifacts; keeping them understates true spend. Don't silently delete — note them. |
| 0-impression + spend | Keep, but flag as **wasted spend** | This is a *client finding*, not noise. Exclude from CVR/CTR math (division by zero). |
| Null `sales` | Keep row for spend analysis; exclude from ACoS | Likely attribution lag on recent days; dropping the row loses valid spend data. |
| Null `bid` | Keep; impute only if a model needs it | Missing bid doesn't invalidate performance metrics. |

The anti-pattern we're screening against is a reflexive `df.dropna()` that quietly discards signal.

In [5]:
# Build an "analysis" frame for efficiency work: valid, real traffic only.
clean = perf[(perf.spend >= 0) & (perf.impressions > 0)].copy()

# Safe ratio helper (avoids divide-by-zero blowups)
def safe_div(a, b):
    return np.where(b > 0, a / np.where(b == 0, np.nan, b), np.nan)

print("Rows kept for efficiency analysis:", len(clean), "of", len(perf))

Rows kept for efficiency analysis: 12544 of 12589


## Part 2.2 — Reshape: keyword × week ACoS + volatility

**The key signal here:** ACoS must be aggregated as **SUM(spend) / SUM(sales)**, computed on the
weekly totals — *not* the mean of per-row ACoS. Averaging ratios over-weights tiny-volume rows and
explodes on zero-sales rows. Watch closely for this.

In [6]:
clean["week"] = clean["date"].dt.to_period("W").dt.start_time

# CORRECT aggregation: sum the components first, THEN divide.
wk = (clean.groupby(["keyword_id", "week"])
            .agg(spend=("spend", "sum"), sales=("sales", "sum"))
            .reset_index())
wk["acos"] = safe_div(wk["spend"], wk["sales"])

# keyword x week matrix of ACoS
mat = wk.pivot(index="keyword_id", columns="week", values="acos")
mat.iloc[:5, :6]

week,2024-04-29,2024-05-06,2024-05-13,2024-05-20,2024-05-27,2024-06-03
keyword_id,,,,,,
5000,NaN,0.422,NaN,NaN,NaN,NaN
5001,NaN,NaN,0.067,NaN,NaN,NaN
5002,NaN,NaN,NaN,NaN,NaN,NaN
5003,NaN,NaN,NaN,NaN,NaN,NaN
5004,NaN,0.028,NaN,NaN,0.092,NaN


In [7]:
# Volatility = week-over-week variation in ACoS.
# Require >=3 weeks of data so sparse keywords don't look artificially volatile.
weeks_present = mat.notna().sum(axis=1)
vol = mat[weeks_present >= 3].std(axis=1).sort_values(ascending=False)

top_vol = vol.head(10).rename("acos_std").reset_index()
top_vol = top_vol.merge(
    clean[["keyword_id", "keyword_text", "match_type"]].drop_duplicates(),
    on="keyword_id", how="left")
print("Most volatile keywords (>=3 weeks of data):")
top_vol

Most volatile keywords (>=3 weeks of data):


,keyword_id,acos_std,keyword_text,match_type
0,5173,0.590,gift kitchen,broad
1,5091,0.570,pro outdoor,broad
2,5128,0.505,heavy duty outdoor,broad
3,5185,0.482,compact kitchen,broad
4,5055,0.476,compact office,phrase
5,5050,0.458,organic pet,broad
6,5037,0.445,compact outdoor,broad
7,5088,0.394,starter outdoor,phrase
8,5108,0.371,pro office,phrase
9,5070,0.342,heavy duty outdoor,phrase


## Part 2.3 — Insight: three things the client should know

Strong candidates form hypotheses, **quantify** them, connect them to an **action**, and flag
**confounders**. Below are the three insights the dataset was designed to contain.

### Insight 1 — Match type drives conversion. Shift budget toward exact match.

In [8]:
by_match = (clean.groupby("match_type")
                  .agg(spend=("spend","sum"), sales=("sales","sum"),
                       clicks=("clicks","sum"), orders=("orders","sum"))
                  .assign(cvr=lambda d: d.orders/d.clicks,
                          acos=lambda d: d.spend/d.sales))
by_match[["cvr","acos","spend","sales"]]

,cvr,acos,spend,sales
match_type,,,,
broad,0.074,0.544,"3,715.360","6,824.700"
exact,0.136,0.261,"4,539.410","17,421.520"
phrase,0.101,0.371,"5,063.050","13,647.220"


Exact match converts markedly better than broad and runs a lower ACoS. **Action:** harvest
high-performing search terms into exact-match campaigns and rebalance spend away from broad.
This maps directly to the harvesting/negation work described in the JD.

*Confounder to name:* match type correlates with keyword intent and bid — this is associative,
not a clean causal estimate. An A/B or holdout would be needed to confirm magnitude.

### Insight 2 — Diminishing returns on bid. Higher bids stop buying conversions.

In [9]:
tmp = clean[clean.clicks > 0].copy()
tmp["bid_bucket"] = pd.qcut(tmp["bid"], 5, duplicates="drop")
bid_curve = (tmp.groupby("bid_bucket", observed=True)
                .agg(cvr=("orders","sum"), clicks=("clicks","sum"),
                     avg_bid=("bid","mean"))
                .assign(cvr=lambda d: d.cvr/d.clicks))
bid_curve[["avg_bid","cvr"]]

,avg_bid,cvr
bid_bucket,,
"(0.099, 0.65]",0.484,0.083
"(0.65, 0.87]",0.771,0.111
"(0.87, 1.06]",0.971,0.106
"(1.06, 1.27]",1.165,0.110
"(1.27, 2.08]",1.505,0.115


CVR rises with bid at first, then flattens. **Action:** cap bids where the marginal conversion
gain plateaus rather than chasing placement. This is exactly the kind of curve the bidding engine
should exploit.

### Insight 3 — Real money is being wasted, and some keywords sit well above target ACoS.

In [10]:
# (a) wasted spend on zero-impression glitch rows
wasted = perf[(perf.impressions == 0) & (perf.spend > 0)].spend.sum()

# (b) keyword-lifetime ACoS vs campaign target
kw = (clean.groupby(["campaign_id","keyword_id","keyword_text"])
           .agg(spend=("spend","sum"), sales=("sales","sum")).reset_index())
kw["acos"] = safe_div(kw["spend"], kw["sales"])
kw = kw.merge(camp[["campaign_id","target_acos","product_category"]], on="campaign_id")
over = kw[(kw.sales > 0) & (kw.acos > kw.target_acos)]

print(f"Wasted spend on zero-impression rows: ${wasted:,.2f}")
print(f"Keywords over target ACoS: {len(over)} of {kw.sales.gt(0).sum()}")
print(f"Spend on over-target keywords: ${over.spend.sum():,.2f}")
over.sort_values('acos', ascending=False).head(8)[
    ['product_category','keyword_text','acos','target_acos','spend']]

Wasted spend on zero-impression rows: $106.92
Keywords over target ACoS: 127 of 189
Spend on over-target keywords: $9,792.99


,product_category,keyword_text,acos,target_acos,spend
118,Pet,refill pet,3.782,0.189,69.810
115,Outdoor,refill outdoor,2.318,0.267,38.830
71,Kitchen,gift kitchen,1.679,0.166,33.980
47,Pet,starter pet,1.634,0.243,100.430
28,Beauty,bulk beauty,1.605,0.340,58.350
51,Pet,compact pet,1.391,0.243,47.740
155,Pet,organic pet,1.255,0.269,145.030
19,Kitchen,organic kitchen,1.243,0.174,47.910


**Action:** lower bids or pause the worst over-target keywords, and fix the billing/tracking
glitch generating spend with no impressions.

**Overall honesty note a strong candidate adds:** recent days may be biased by attribution lag
(sales still landing), so the most recent week's ACoS looks worse than it truly is.

## Part 3 — Modeling: will a keyword-day convert?

Frame as **binary classification**: `converts = orders > 0`. We show Logistic Regression (baseline,
interpretable) and XGBoost (the JD names both). The two decisions that matter most are the
**time-based split** and **leakage-aware features**.

In [11]:
model_df = perf[(perf.spend >= 0) & (perf.clicks > 0)].copy()
model_df["converts"] = (model_df["orders"] > 0).astype(int)
print("Rows:", len(model_df), "| conversion rate: %.3f" % model_df.converts.mean())
print("=> imbalanced-ish, so accuracy is the WRONG metric; use AUC / PR-AUC.")

Rows: 7142 | conversion rate: 0.171
=> imbalanced-ish, so accuracy is the WRONG metric; use AUC / PR-AUC.


**Leakage discipline (call this out in the interview):**
- ❌ `sales` — derived from orders; using it to predict conversion is circular.
- ❌ `orders` — the label.
- ✅ `bid`, `impressions`, `clicks`, `match_type`, `product_category` — known at/near bid time.
  (`clicks` precede `orders`, so they're legitimate predictors.)

In [12]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score, average_precision_score, classification_report
from xgboost import XGBClassifier

model_df = model_df.merge(camp[["campaign_id","product_category"]], on="campaign_id",
                          how="left", suffixes=("","_c"))

num_feats = ["bid", "impressions", "clicks"]
cat_feats = ["match_type", "product_category"]
features  = num_feats + cat_feats

# impute the few null bids with the median (fit on train only, via the pipeline in practice)
model_df["bid"] = model_df["bid"].fillna(model_df["bid"].median())

# --- THE CRITICAL DECISION: split by TIME, not randomly ---
# Random splits leak the future into the past on time-series data.
cutoff = model_df["date"].quantile(0.8)
train = model_df[model_df.date <= cutoff]
test  = model_df[model_df.date >  cutoff]
print(f"Train <= {cutoff.date()}: {len(train)} rows | Test: {len(test)} rows")

X_train, y_train = train[features], train["converts"]
X_test,  y_test  = test[features],  test["converts"]

Train <= 2024-06-18: 5783 rows | Test: 1359 rows


In [13]:
pre = ColumnTransformer([
    ("num", StandardScaler(), num_feats),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_feats),
])

logit = Pipeline([("pre", pre),
                  ("clf", LogisticRegression(max_iter=1000, class_weight="balanced"))])
logit.fit(X_train, y_train)
p_logit = logit.predict_proba(X_test)[:, 1]

print("Logistic Regression")
print("  ROC-AUC : %.3f" % roc_auc_score(y_test, p_logit))
print("  PR-AUC  : %.3f" % average_precision_score(y_test, p_logit))

Logistic Regression
  ROC-AUC : 0.686
  PR-AUC  : 0.352


In [14]:
xgb = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9, eval_metric="logloss",
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    random_state=42)

# XGBoost handles categoricals via one-hot here for parity with the linear model
Xtr = pre.fit_transform(X_train); Xte = pre.transform(X_test)
xgb.fit(Xtr, y_train)
p_xgb = xgb.predict_proba(Xte)[:, 1]

print("XGBoost")
print("  ROC-AUC : %.3f" % roc_auc_score(y_test, p_xgb))
print("  PR-AUC  : %.3f" % average_precision_score(y_test, p_xgb))
print()
print(classification_report(y_test, (p_xgb > 0.5).astype(int), digits=3))

XGBoost
  ROC-AUC : 0.659
  PR-AUC  : 0.336

              precision    recall  f1-score   support

           0      0.888     0.719     0.795      1132
           1      0.281     0.546     0.371       227

    accuracy                          0.690      1359
   macro avg      0.584     0.633     0.583      1359
weighted avg      0.786     0.690     0.724      1359



In [15]:
# Feature importance sanity check — do the drivers match the insights?
ohe_names = pre.named_transformers_["cat"].get_feature_names_out(cat_feats)
feat_names = num_feats + list(ohe_names)
imp = (pd.Series(xgb.feature_importances_, index=feat_names)
         .sort_values(ascending=False).head(10))
imp

clicks                         0.269
match_type_exact               0.087
match_type_broad               0.083
bid                            0.073
product_category_Pet           0.071
impressions                    0.069
product_category_Office        0.066
product_category_Supplements   0.063
product_category_Kitchen       0.062
product_category_Beauty        0.057
dtype: float32

## Part 3 — "Would you ship it?"

The JD is explicit that this engine **takes autonomous action** — it sets bids and moves real
budget, it doesn't just recommend. So the answer we want is **not** "AUC looks fine, ship it." A
strong candidate says something like:

- **Not straight to production.** Start in **shadow mode** (log what it *would* bid) or a small
  **canary** slice of traffic, and compare against the current engine before widening.
- **Guardrails**: hard bid caps, max daily spend deltas, and a kill-switch — a bad prediction here
  spends money, not just misfiles a row.
- **Monitor for drift**: feature and prediction distributions shift as the market moves; retrain
  cadence + alerting on AUC/PR-AUC decay on rolling holdouts.
- **Watch the leakage boundary in production**: features must be available *at bid time*, not
  backfilled after attribution.
- **Business metric, not just ML metric**: ultimately judged on ACoS-vs-target and incremental
  sales, via a holdout/geo experiment — not offline AUC alone.

That connection — from an offline model to a *cautious, monitored, guard-railed* autonomous system
— is the strongest senior signal in the whole exercise.